# Part 5c: RTL co-simulation

C simulation (Part 5b) only checks the C++. **RTL co-simulation** (`cosim`) goes one step further: Vitis HLS synthesizes the C++ to RTL and then simulates the RTL with the same testbench, comparing the results against the C++ model. This is the last check before spending hours on a bitstream, and it is the stage where interface problems (handshakes, stream widths, FIFO depths) show up.

Both `axi_master` and `axi_stream` modes are supported.

## Settings

Set the target board here. The backend picks the platform (`.xpfm` or `.xsa`) from its `supported_boards.json` using the board and the AXI mode, so nothing else needs to be configured. To add another board, see `5_vu_f_platform_setup.md`.

| Variable | Meaning |
|----------|---------|
| `BOARD` | `'zcu102'` or `'kv260'` |
| `AXI_MODE` | `'axi_master'` (default) or `'axi_stream'`; see the *AXI interface modes* page of the hls4ml docs |

The Vitis tools are located through the `XILINX_VITIS` environment variable, as in the other parts of this tutorial.

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import sys

sys.path.append('../..')
from models import create_simple_unet
import hls4ml

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

BOARD = 'zcu102'  # or 'kv260'
AXI_MODE = 'axi_master'  # or 'axi_stream'

DATA_DIR = '../../data/vitis_unified'
os.makedirs(DATA_DIR, exist_ok=True)
np.random.seed(0)

## Create the model and some test data

We use a tiny U-Net with a single skip connection, defined in `models.py`. The weights are left untrained on purpose: the goal of this part is to check that every stage of the accelerator flow reproduces the same numbers, not to reach a good accuracy. A `Concatenate` skip connection is a good stress test for the streaming interface, because it forces two branches to be merged again.

In [ ]:
X = np.random.rand(10, 4, 4, 1).astype(np.float32)
np.save(f'{DATA_DIR}/X_part5c.npy', X)
print(f'Input shape: {X.shape}')

model = create_simple_unet()
model.summary()

## Reference predictions for the testbench

The testbench generated by hls4ml reads its inputs and expected outputs from files. We pass them with `input_data_tb` and `output_data_tb`. The expected outputs are produced with the C++ bridge, exactly as in Part 5a, so the comparison checks the tool flow rather than the model.

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')

# Reference predictions from the C++ bridge
hls_model_ref = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='VitisUnified',
    io_type='io_stream',
    output_dir=os.path.abspath(f'../../hls4ml_prjs/hls4ml_prj_part5c_ref_{AXI_MODE}'),
    board=BOARD,
    axi_mode=AXI_MODE,
    input_type='float',
    output_type='float',
    clock_period=10,
)
hls_model_ref.compile()
y_ref = hls_model_ref.predict(X)
np.save(f'{DATA_DIR}/y_part5c.npy', y_ref)
print(f'Reference predictions saved: shape {np.array(y_ref).shape}')

## Build and run the RTL co-simulation

`build(cosim=True)` synthesizes the design and then simulates the generated RTL. **This can take 10-20 minutes** even for this tiny model, since an RTL simulation runs cycle by cycle.

In [ ]:
output_dir = os.path.abspath(f'../../hls4ml_prjs/hls4ml_prj_part5c_{AXI_MODE}')

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='VitisUnified',
    io_type='io_stream',
    output_dir=output_dir,
    board=BOARD,
    axi_mode=AXI_MODE,
    input_type='float',
    output_type='float',
    input_data_tb=f'{DATA_DIR}/X_part5c.npy',
    output_data_tb=f'{DATA_DIR}/y_part5c.npy',
    clock_period=10,
)
hls_model.compile()

In [ ]:
hls_model.build(synth=True, cosim=True, log_to_stdout=True)

## Compare results

The RTL result must match the bridge prediction within `1e-4`.

In [ ]:
y_bridge = np.loadtxt(f'{output_dir}/tb_data/tb_output_predictions.dat')
y_cosim = np.loadtxt(f'{output_dir}/tb_data/rtl_cosim_results.log')

print(f'Bridge shape: {y_bridge.shape}')
print(f'CoSim  shape: {y_cosim.shape}')

assert np.allclose(y_bridge, y_cosim, rtol=0.0, atol=1e-4), 'The results from bridge and cosim are NOT equal!'
print('RTL co-simulation comparison passed (atol=1e-4)')